In [4]:
!pip install google-play-scraper


In [5]:
!pip install langchain langchain-community langchain-openai


In [6]:
import os
os.environ["OPENAI_API_KEY"] = "sk-or-v17"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"


In [7]:
import os
import pandas as pd

In [ ]:
from google_play_scraper import reviews, Sort

APP_ID = "com.application.zomato"

result, _ = reviews(
    APP_ID,
    lang='en',
    country='in',
    sort=Sort.NEWEST,
    count=50
)

for r in result:
    print(f"Date: {r.get('at')}")
    print(f"Rating: {r.get('score')}")
    print(f"Review: {r.get('content')}")
    print("-" * 10)



Date: 2026-01-09 09:46:31
Rating: 5
Review: excellent
----------
Date: 2026-01-09 09:46:28
Rating: 5
Review: zomato is a very good app for food lover...service also very good...
----------
Date: 2026-01-09 09:46:01
Rating: 4
Review: express delivery
----------
Date: 2026-01-09 09:45:31
Rating: 5
Review: A low-priced, simple UI.
----------
Date: 2026-01-09 09:44:41
Rating: 5
Review: nice app
----------
Date: 2026-01-09 09:43:59
Rating: 5
Review: good
----------
Date: 2026-01-09 09:42:36
Rating: 5
Review: expensive service and time actually fix
----------
Date: 2026-01-09 09:40:56
Rating: 5
Review: nicee
----------
Date: 2026-01-09 09:39:29
Rating: 5
Review: Great
----------
Date: 2026-01-09 09:37:50
Rating: 4
Review: good
----------
Date: 2026-01-09 09:35:54
Rating: 5
Review: Happy
----------
Date: 2026-01-09 09:35:34
Rating: 5
Review: As per my experience Zomato is very useful for me. Keep updating. Thanks.
----------
Date: 2026-01-09 09:32:23
Rating: 5
Review: good service 👍 we are re

In [42]:
df=pd.DataFrame(result)
df['date'] = df['at'].dt.date
df[['date', 'score', 'content']].head()


,date,score,content
0,2026-01-09,5,excellent
1,2026-01-09,5,zomato is a very good app for food lover...ser...
2,2026-01-09,4,express delivery
3,2026-01-09,5,"A low-priced, simple UI."
4,2026-01-09,5,nice app


In [43]:
base_path = "data/zomato"
os.makedirs(base_path, exist_ok=True)


In [44]:
for date, daily_df in df.groupby('date'):
    file_path = f"{base_path}/{date}.csv"
    daily_df[['date', 'score', 'content']].to_csv(file_path, index=False)


In [49]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=128
)




In [50]:
df = pd.read_csv("/content/data/zomato/2026-01-08.csv")

sample_review = df.iloc[0]["content"]

print(sample_review)


excellent


In [51]:
prompt = PromptTemplate(template="""
You are analyzing Google Play Store app reviews.

Your task is to extract ALL concrete topics related to:
- specific issues
- specific complaints
- specific requests
- specific suggestions
- specific feedback about features, service, or experience

CRITICAL RULES:
- Topics MUST refer to a concrete aspect (delivery, food, app, support, pricing, etc.)
- Do NOT return generic phrases like:
  "Positive feedback", "Negative feedback", "Good experience", "Bad experience"
- Do NOT summarize sentiment
- Topics must be short, specific phrases (3–6 words)
- Always extract at least one concrete topic
- Multiple topics are allowed

Review:
{review}

Return topics as a bullet list.
""")



In [52]:
chain=prompt | model | StrOutputParser()

response = chain.invoke({
    "review": sample_review
})

print(response)


- No specific topics identified


In [53]:
canonical_topics = set([
    "Delivery issue",
    "Delivery partner rude",
    "Late delivery",
    "Food quality issue",
    "App performance issue",
    "App user interface"
])


In [54]:
canonical_prompt = PromptTemplate(template="""
You are a topic canonicalization agent for app review analysis.

Your task:
Given a newly extracted topic and a list of existing canonical topics,
decide the BEST canonical topic.

Rules:
- Merge aggressively
- If the new topic is even loosely similar to an existing topic, MERGE
- Prefer existing topics over creating new ones
- Canonical topics must be short, neutral, and stable over time
- Avoid synonyms, paraphrases, or stylistic variations

If merging:
- Return the EXACT existing topic name

If creating a new topic:
- Rewrite it into a clean, neutral canonical form

Inputs:
New topic:
{new_topic}

Existing canonical topics:
{existing_topics}

Return ONLY ONE topic name. No explanation.
""")


In [55]:
canonical_chain = canonical_prompt | model | StrOutputParser()

testing with real eg


In [56]:
existing = list(canonical_topics)

response = canonical_chain.invoke({
    "new_topic": "Smooth performance",
    "existing_topics": existing
})

print(response)


App performance issue


In [57]:
response = canonical_chain.invoke({
    "new_topic": "UI design",
    "existing_topics": existing
})

print(response)


App user interface


In [58]:
response = canonical_chain.invoke({
    "new_topic": "Refund not received",
    "existing_topics": existing
})

print(response)


Refund not received


In [59]:
canonical_topics.add(response)


In [60]:
canonical_topics

{'App performance issue',
 'App user interface',
 'Delivery issue',
 'Delivery partner rude',
 'Food quality issue',
 'Late delivery',
 'Refund not received'}

In [61]:
topic_chain = chain


In [62]:
import os

data_dir = "/content/data/zomato"
daily_topic_store = {}

for file in sorted(os.listdir(data_dir)):
    if file.endswith(".csv"):
        date = file.replace(".csv", "")
        print("Processing:", date)

        daily_topic_store[date] = process_daily_reviews(
            csv_path=f"{data_dir}/{file}",
            topic_chain=topic_chain,
            canonical_chain=canonical_chain,
            canonical_topics=canonical_topics
        )


Processing: 2026-01-08


Processing reviews: 100%|██████████| 35/35 [01:40<00:00,  2.88s/it]


Processing: 2026-01-09


Processing reviews: 100%|██████████| 50/50 [01:46<00:00,  2.13s/it]


In [28]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [65]:
trend_df = pd.DataFrame(daily_topic_store).fillna(0).astype(int)
trend_df


,2026-01-08,2026-01-09
App performance issue,5,4
No specific suggestions mentioned,82,131
App user interface,11,8
Delivery issue,7,7
Late delivery,3,5
Food quality issue,7,2
Refund not received,1,0
Delivery partner rude,0,1


In [67]:

os.makedirs("output", exist_ok=True)

In [69]:
output_path = "output/zomato_topic_trend_sample.csv"
trend_df.to_csv(output_path)


